# Solutions · 13 · Diagnosing failures

Worked solutions to the exercises of [notebook 13](../notebooks/13_diagnosing_failures.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, stiffness, analysis, grids, boundary as bc, operators as op, spectral
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Take the heat equation with Neumann ends and impose the Neumann condition by a *first-order* one-sided ghost (u_ghost = u_0 - dx g) instead of the centred one. Measure the global order of accuracy and the conservation error. Which diagnostic reveals the problem first?

In [2]:
def heat_neumann_firstorder(n):
    g = grids.uniform(0, 1, n); dx = g.dx
    def rhs(t, u):
        up = np.r_[u[0], u, u[-1]]                     # first-order ghosts: u_ghost = u_0 (zero slope by a one-sided difference)
        return (up[2:] - 2 * up[1:-1] + up[:-2]) / dx**2
    return g, rhs
errs, cons = [], []
for n in (20, 40, 80):
    g, rhs = heat_neumann_firstorder(n)
    u0 = np.cos(np.pi * g.x)
    s = ti.bdf(rhs, u0, (0, 0.2), 1e-3, 4, band=(1, 1), n_out=1)
    errs.append(np.max(np.abs(s.y[-1] - np.cos(np.pi * g.x) * np.exp(-np.pi**2 * 0.2))))
    cons.append(abs(g.integrate(s.y[-1]) - g.integrate(u0)))
print("errors:", ["%.2e" % e for e in errs], "orders", np.round(analysis.observed_order(errs), 2), "(the centred ghost gives 2)")
print("conservation errors:", ["%.1e" % c for c in cons], "(the centred ghost gives 1e-16)")

errors: ['3.20e-02', '1.57e-02', '7.80e-03'] orders [1.02 1.01] (the centred ghost gives 2)
conservation errors: ['4.2e-16', '1.5e-16', '1.1e-16'] (the centred ghost gives 1e-16)


The first-order ghost makes the whole solution first-order accurate - the boundary error dominates - and,
more visibly, it breaks conservation: the flux through the end is no longer exactly zero, so the heat content
drifts. The conservation check reveals the problem from the first run; the order study needs three.

## Exercise 2

Integrate Fisher's equation with an explicit step just above the diffusion limit. Describe the symptom: where does the instability start, and how fast does it grow per step (compare with |R(h lambda_max)|)?

In [3]:
p = pr.fisher(n=200)
lam = stiffness.spectrum(p.rhs, p.y0, band=(1, 1)); lam = lam[np.abs(lam) > 1e-9]
h_lim = stiffness.explicit_step_limit(lam, "rk4")
with np.errstate(all="ignore"):
    s = ti.explicit_rk(p.rhs, p.y0, (0, 60 * 1.05 * h_lim), 1.05 * h_lim, n_out=6)
growth = [np.max(np.abs(y - p.y0)) for y in s.y]
print(f"RK4 at 1.05 x the limit (h = {1.05 * h_lim:.4f}): deviation from the initial state every 10 steps {['%.1e' % g for g in growth]}")
R = abs(stiffness.stability_function("rk4", 1.05 * h_lim * lam[0].real))
print(f"|R(h lambda_max)| = {R:.3f} -> expected growth per 10 steps {R**10:.2f}x; the sawtooth appears first where the solution is roughest (the step's edge)")

RK4 at 1.05 x the limit (h = 0.0645): deviation from the initial state every 10 steps ['0.0e+00', '5.5e-01', '9.9e-01', '2.6e+00', '9.7e+00', 'nan', 'nan']
|R(h lambda_max)| = 1.231 -> expected growth per 10 steps 8.00x; the sawtooth appears first where the solution is roughest (the step's edge)


Just above the limit the sawtooth mode grows by |R(hλ_max)| = 1.23 per step, eightfold per ten steps once
it dominates; at first the deviation shown is mostly the physical evolution of the front, and the geometric
growth takes over after a few dozen steps and ends in overflow. The sawtooth is seeded where the solution has
the most high-frequency content, at the step in the initial data. The symptom - a grid-scale oscillation
growing geometrically, independent of the physics - identifies a time-step instability as surely as the
spectrum does.

## Exercise 3

**Implement it yourself:** write a function `diagnose(rhs, y, band)` that reports the spectrum's largest real part, the stiffness ratio, the RK4 and Euler step limits, and whether the spectrum touches the imaginary axis; run it on five problems of the library.

In [4]:
def diagnose(rhs, y, band=None):
    lam = stiffness.spectrum(rhs, y, band=band); lam = lam[np.abs(lam) > 1e-9]
    decaying = lam[lam.real < 0]
    return {"max Re": float(lam.real.max()), "|lambda| ratio": float(np.abs(lam).max() / np.abs(lam).min()), "RK4 limit": stiffness.explicit_step_limit(decaying, "rk4"),
            "Euler limit": stiffness.explicit_step_limit(decaying, "euler"), "imaginary-axis modes": bool(np.any((np.abs(lam.real) < 1e-6) & (np.abs(lam.imag) > 1e-6)))}
rows = []
for make in (pr.heat_dirichlet, pr.advection_periodic, pr.fisher, pr.wave_string, pr.cahn_hilliard):
    p = make()
    d = diagnose(p.rhs, p.y0, p.band)
    rows.append((p.name, *d.values()))
print(pd.DataFrame(rows, columns=["problem", "max Re", "|lambda| ratio", "RK4 limit", "Euler limit", "imaginary modes"]).to_string(index=False, float_format="%.3g"))

                              problem   max Re  |lambda| ratio  RK4 limit  Euler limit  imaginary modes
                 heat, Dirichlet ends    -9.86             648   0.000436     0.000313            False
linear advection, periodic (van-leer) -0.00101        3.62e+05    0.00761      0.00176            False
           Fisher-KPP travelling wave    0.999        6.04e+03     0.0156       0.0112            False
                     vibrating string 8.88e-14            63.7     0.0142     7.08e-09             True
       Cahn-Hilliard phase separation      613        1.07e+12   1.68e-06     1.21e-06            False


One function, five diagnoses: the heat equation is stiff and real; the string sits on the imaginary axis (no
Euler step is stable; RK4's is set by the CFL condition) and the limited advection scheme's nonlinear
Jacobian sits near it; Fisher has a positive real part (the unstable state u = 0 that the front invades, a
physical instability); Cahn-Hilliard, linearised about a random mixture, has large positive eigenvalues - the
spinodal instability that drives phase separation - and is extremely stiff. Running such a function before
choosing an integrator is the discipline of the course.